# Analiza sentymentu komentarzy o kryptowalutach – dostrajanie CryptoBERT

Notebook korzysta z funkcji w `src/` – tych samych, które uruchamia potok DVC (`dvc repro`), CI i Jenkins.
Parametry są w `params.yaml`, przebiegi trafiają do MLflow na DagsHub (konfiguracja w pliku `.env`).

In [ ]:
import os
import sys
from pathlib import Path

# katalog główny projektu (notebook leży w notebooks/)
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

import torch

from src.params import load_params, override, save_params

params = load_params()
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "brak (CPU)")
params

## 1. Budowa zbioru danych (opcjonalnie)

Gotowy zbiór `data/final_crypto_data.csv` jest wersjonowany w DVC (`dvc pull`). Poniższe kroki są potrzebne
tylko do zbudowania nowej wersji – wymagają kluczy Reddit API w `.env` i trwają długo.

In [ ]:
BUILD_DATASET = False

if BUILD_DATASET:
    from src.dataset import augment, filter_comments, pseudo_label, scrape_reddit

    raw = scrape_reddit("CryptoCurrency", limit=1000)                    # pobranie danych
    clean = filter_comments(raw)                                         # czyszczenie i filtrowanie
    labeled = pseudo_label(clean)                                        # głosowanie 3 modeli
    final = augment(labeled, n=3)                                        # augmentacja synonimami
    final.to_csv(params["data"]["input"], index=False)
    # nowa wersja zbioru: dvc add data/final_crypto_data.csv && dvc push

## 2. Przygotowanie danych (etap `prepare`)

In [ ]:
from src.data import prepare

splits = prepare(params)
splits["train"]["label"].value_counts().rename(index={0: "negative", 1: "neutral", 2: "positive"})

## 3. Poszukiwanie hiperparametrów (opcjonalnie)

Odpowiednik sweepa W&B: Optuna + MLflow – każda próba to zagnieżdżony przebieg w eksperymencie.
Najlepsze parametry można zapisać do `params.yaml`.

In [ ]:
RUN_SWEEP = False

if RUN_SWEEP:
    from src.sweep import run_sweep

    best = run_sweep(params, n_trials=9)
    params = override(params, [f"train.{k}={v}" for k, v in best.items()])
    save_params(params)

## 4. Dostrajanie (etap `train`)

Metryki z każdej epoki są logowane do MLflow, a model zapisywany w `models/model`.

In [ ]:
from src.train import train

result = train(params)
result["history"]

In [ ]:
import matplotlib.pyplot as plt

history = result["history"]
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
history.plot(x="epoch", y=["train_loss", "eval_loss"], marker="o", ax=axes[0], title="Strata")
history.plot(x="epoch", y=["eval_accuracy", "eval_f1"], marker="o", ax=axes[1], title="Metryki walidacyjne")
plt.show()

## 5. Ewaluacja na zbiorze testowym (etap `evaluate`)

In [ ]:
from IPython.display import Image, display

from src.evaluate import evaluate

metrics = evaluate(params)
display(Image("reports/confusion_matrix.png"))
metrics

In [ ]:
import pandas as pd

# błędnie sklasyfikowane przykłady
pd.read_csv("reports/misclassified.csv").head(10)

## 6. Rejestracja modelu w MLflow Model Registry (opcjonalnie)

Rejestruje model jako `cryptobert-sentiment` z aliasem `champion` (w Jenkinsie robi to potok po progu jakości).

In [ ]:
REGISTER = False

if REGISTER:
    from src.register import register

    register()

## 7. Predykcja – tak samo jak w API

In [ ]:
from src.predict import Predictor

predictor = Predictor("models/model", max_length=params["train"]["max_length"])
predictor.predict([
    "bitcoin is going to the moon, best investment ever",
    "eth gas fees are killing me, this is a scam",
    "just bought some btc today",
])